# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.


In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)

# The mlcroissant Dataset.metadata object provides a direct way to access dataset metadata attributes
print(f"{dataset.metadata.name}: {dataset.metadata.description}\n")
print(f"Published on: {dataset.metadata.date_published}")
print(f"Keywords: {dataset.metadata.keywords}")
print(f"License: {dataset.metadata.license}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

Here, we enumerate the record sets and their associated field and column `@id`s, available in this Croissant dataset.

In [ ]:
# List all record sets in the dataset, referencing by their @id
record_sets = list(dataset.record_sets)
if not record_sets:
    print("No record sets found in the dataset metadata.")
else:
    for rs in record_sets:
        print(f"Record Set: @id={rs['@id']}")
        print(f"  - name: {rs.get('name', '')}")
        print(f"  - description: {rs.get('description', '')}")
        # Show field and column @ids (fields of this record set)
        fields = rs.get('field', [])
        if isinstance(fields, dict):
            fields = [fields]
        if fields:
            print(f"  - fields: {[f['@id'] if isinstance(f, dict) else f for f in fields]}")
        columns = rs.get('column', [])
        if isinstance(columns, dict):
            columns = [columns]
        if columns:
            print(f"  - columns: {[c['@id'] if isinstance(c, dict) else c for c in columns]}")
        print("")
    # Optionally, print the first few records from the first record set for inspection
    sample_set = record_sets[0]['@id']
    print(f"--- Example records from record set {sample_set} ---")
    for ix, rec in enumerate(dataset.records(record_set=sample_set)):
        print(rec)
        if ix >= 2:
            break

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview above.

In [ ]:
# If there are no record sets, this block will not run correctly
# In this dataset, we will try to find the main record set:
record_sets = list(dataset.record_sets)
dataframes = {}
record_set_ids = [rs['@id'] for rs in record_sets] if record_sets else []
if not record_set_ids:
    print("No record sets defined in this dataset. \nCannot extract tabular data without record set definitions.")
else:
    for record_set_id in record_set_ids:
        records = list(dataset.records(record_set=record_set_id))
        dataframes[record_set_id] = pd.DataFrame(records)
    # Show dataframe columns for the first record set
    first_rs = record_set_ids[0]
    print(f"Columns for record set {first_rs}:")
    print(dataframes[first_rs].columns.tolist())
    dataframes[first_rs].head()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section includes operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.


In [ ]:
# Let's perform EDA on the first available record set, referencing fields by their @id
if not record_set_ids:
    print("No record sets available for EDA.")
else:
    df = dataframes[first_rs].copy()
    print(f"Dataframe shape: {df.shape}")

    # Choose a numeric field for analysis, if any exist
    # Let's try to auto-detect a numeric field
    numeric_field = None
    for col in df.columns:
        # Make sure '@id' is used as field references
        if pd.api.types.is_numeric_dtype(df[col]):
            numeric_field = col
            break
    if not numeric_field:
        print("No numeric field found for analysis.")
    else:
        threshold = df[numeric_field].quantile(0.75)
        filtered_df = df[df[numeric_field] > threshold]
        print(f"Filtered records where {numeric_field} > {threshold:.2f}:")
        print(filtered_df.head())

        # Normalize the numeric field
        filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        print(f"Normalized {numeric_field} for filtered records:")
        print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

        # Attempt grouping if any object columns are present
        group_field = None
        for col in df.columns:
            if pd.api.types.is_object_dtype(df[col]) and col != numeric_field:
                group_field = col
                break
        if group_field:
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index().sort_values(by=numeric_field, ascending=False)
            print(f"Grouped data by {group_field} (mean {numeric_field}):")
            print(grouped_df.head())
        else:
            print("No suitable grouping (categorical) field found.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt

if not record_set_ids or not numeric_field:
    print("No numeric field available for visualization.")
else:
    # Histogram of the numeric field
    plt.figure(figsize=(8,4))
    df[numeric_field].hist(bins=30)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel('Count')
    plt.show()
    # If grouped data is available, show a bar plot
    if 'grouped_df' in locals():
        plt.figure(figsize=(8,4))
        plt.bar(grouped_df[group_field].astype(str), grouped_df[numeric_field])
        plt.title(f"Mean {numeric_field} by {group_field}")
        plt.xlabel(group_field)
        plt.ylabel(f"Mean {numeric_field}")
        plt.xticks(rotation=45)
        plt.tight_layout()
        plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.


<!-- You can add your concluding remarks here -->